# CRISP-DM Project: Predicting 30-Day Diabetic Hospital Readmission

This notebook follows all six CRISP-DM phases. It is designed to run in Google Colab or locally.

**Responsible-use statement:** This is an educational prototype using historical, de-identified data. It is not a medical device and must not be used for diagnosis or treatment decisions.

## 1. Business Understanding

Hospitals want to identify diabetic encounters that may have elevated risk of readmission within 30 days. A care-management team could use a risk score to prioritize discharge planning and follow-up resources.

**Business objective:** support targeted post-discharge follow-up.

**Data-mining objective:** estimate the probability that an encounter will be followed by readmission within 30 days.

**Primary success criteria:** report PR-AUC and recall/precision at an explicit operating threshold, while checking calibration, leakage, and subgroup performance. Accuracy alone is insufficient because the positive class is uncommon.

**Constraints:** historical data, missing clinical fields, repeated patients, class imbalance, and no claim of clinical validity.

## 2. Data Understanding

The dataset is the Diabetes 130-US Hospitals dataset, originally collected from 130 US hospitals and integrated delivery networks over 1999–2008. The target is `readmitted`, where `<30` represents readmission within 30 days.

Place `diabetic_data.csv` in `data/raw/` locally or upload it to the Colab session as `/content/diabetic_data.csv`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve, roc_auc_score,
                             RocCurveDisplay)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
LOCAL_PATH = Path('data/raw/diabetic_data.csv')
COLAB_PATH = Path('/content/diabetic_data.csv')
DATA_PATH = COLAB_PATH if COLAB_PATH.exists() else LOCAL_PATH
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Place diabetic_data.csv at {LOCAL_PATH} or upload it to {COLAB_PATH}')
df_raw = pd.read_csv(DATA_PATH, na_values=['?'])
print(f'Loaded: {DATA_PATH}')
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
print(df_raw.info())
display(df_raw.describe(include='all').T.head(20))
missing = (df_raw.isna().mean().sort_values(ascending=False) * 100).round(2)
display(missing[missing > 0].to_frame('missing_percent'))

plt.figure(figsize=(7, 4))
sns.countplot(data=df_raw, x='readmitted', order=['NO', '>30', '<30'])
plt.title('Original readmission labels')
plt.show()

### Data-quality observations

We will document the actual output after execution. Particular attention is required for: placeholder missing values, high-missingness columns, repeated patients, identifier columns, and the imbalanced `<30` target.

## 3. Data Preparation

The target is converted to binary: `<30` becomes 1 and all other readmission labels become 0. Patient identifiers are retained only for grouped splitting and are not used as predictors. The split is performed by patient so the same patient does not appear in both training and test data.

In [ ]:
df = df_raw.copy()
y = (df['readmitted'].astype(str) == '<30').astype(int)
groups = df['patient_nbr'].copy()

drop_columns = ['readmitted', 'encounter_id', 'patient_nbr', 'weight',
                 'payer_code', 'medical_specialty']
X = df.drop(columns=[c for c in drop_columns if c in df.columns]).copy()
if 'age' in X.columns:
    X['age'] = X['age'].astype('string')

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Train positive rate:', round(y_train.mean(), 4))
print('Test positive rate:', round(y_test.mean(), 4))

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features),
])
print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

## 4. Modeling

We compare an interpretable logistic-regression baseline with a random forest. Class weighting is used because the positive class is imbalanced. All transformations are inside the pipeline to prevent test-set information from influencing training.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5,
        class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1
    ),
}

fitted_models = {}
results = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', estimator)])
    pipe.fit(X_train, y_train)
    probability = pipe.predict_proba(X_test)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    report = classification_report(y_test, prediction, output_dict=True, zero_division=0)
    results.append({
        'model': name,
        'roc_auc': roc_auc_score(y_test, probability),
        'pr_auc': average_precision_score(y_test, probability),
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False)
display(results_df)

## 5. Evaluation

The positive class is readmission within 30 days. We inspect both ranking metrics and threshold-dependent metrics. A threshold should ultimately be selected with stakeholders based on the relative cost of missed high-risk encounters and unnecessary interventions.

In [ ]:
best_name = results_df.iloc[0]['model']
best_model = fitted_models[best_name]
best_probability = best_model.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
RocCurveDisplay.from_predictions(y_test, best_probability, name=best_name, ax=axes[0])
axes[0].set_title(f'ROC curve: {best_name}')
precision, recall, thresholds = precision_recall_curve(y_test, best_probability)
axes[1].plot(recall, precision)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-recall curve: {best_name}')
plt.tight_layout()
plt.show()

best_prediction = (best_probability >= 0.5).astype(int)
print(classification_report(y_test, best_prediction, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, best_prediction, cmap='Blues')
plt.title(f'Confusion matrix: {best_name}')
plt.show()

In [ ]:
threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.05):
    pred = (best_probability >= threshold).astype(int)
    report = classification_report(y_test, pred, output_dict=True, zero_division=0)
    threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'precision': report['1']['precision'],
        'recall': report['1']['recall'],
        'f1': report['1']['f1-score'],
        'flagged_rate': pred.mean(),
    })
threshold_df = pd.DataFrame(threshold_rows)
display(threshold_df)

## 6. Deployment

A responsible deployment would score eligible encounters at discharge, route high-risk cases to a care-management queue, and require human review before any intervention. It would monitor:

- input missingness and category drift;
- calibration and PR-AUC over time;
- recall and precision at the selected threshold;
- performance across available demographic groups;
- intervention capacity and false-positive burden.

The model should be retrained or withdrawn if data quality or performance deteriorates. Before real-world use, external validation, clinical review, privacy review, and prospective impact assessment would be required.